# cordslite API

In [ ]:
#| default_exp core

In [ ]:
#| export
from datetime import datetime, timedelta, timezone
from fasthtml.common import *
from fastcore.meta import *
from fastcore.utils import *
from nacl.bindings import crypto_aead_xchacha20poly1305_ietf_decrypt as xchacha_decrypt

import asyncio,ffmpeg,httpx,json,opuslib_next,os,random,re,socket,time
import websockets.asyncio.client

In [ ]:
from IPython.display import Audio

Discord has three main APIs:
- **REST API** - for actions (send message, get channels, fetch history). Request/response style.
- **Gateway API** - for real-time events via WebSocket (new messages, reactions, user joins).
- **Voice API** - for real-time streaming of audio via UDP

We start with the REST API. To use it, you need a **bot token** from the [Discord Developer Portal](https://discord.com/developers/applications):
1. Create an application
2. Go to "Bot" section and create a bot
3. Copy the token (keep it secret!)
4. Under "OAuth2 → URL Generator", select `bot` scope and choose permissions (e.g., `Send Messages`, `Read Message History`)
5. Use the generated URL to invite the bot to your server

The `DiscordClient` wraps `httpx.Client` with Discord's base URL (`https://discord.com/api/v10`) and auth headers pre-configured.

In [ ]:
#| export
class DiscordClient:
    def __init__(self, token=None, user_token=None, name='cordslite', ver='0.1'):
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.user_token = user_token or os.environ.get('DISCORD_USER_TOKEN')
        self.base_url = 'https://discord.com/api/v10'
        self.headers = {'Authorization': f'Bot {self.token}', 'User-Agent': f'DiscordBot ({name}, {ver})'}
        self.cli = httpx.AsyncClient(base_url=self.base_url, headers=self.headers)


The token defaults to the `DISCORD_BOT_TOKEN` environment variable—no hardcoding secrets!

In [ ]:
dc = DiscordClient()


## REST endpoints

Every REST call goes through `_req`, which handles two concerns automatically:
- **Rate limiting** — if Discord returns `429 Too Many Requests`, we wait the `Retry-After` duration and retry.
- **Error handling** — non-2xx responses are raised as `DiscordError` with Discord's error code, message, and HTTP status, so callers don't need to check responses manually.

In [ ]:
#| export
class DiscordError(Exception):
    def __init__(self, code, msg, status): self.code,self.msg,self.status = code,msg,status
    def __str__(self): return f"DiscordError({self.status}): [{self.code}] {self.msg}"

@patch
async def _req(self:DiscordClient, method, path, data=None, files=None, use_user=False, **kw):
    kw = filter_values(kw, negate(is_(None)))
    if method=='GET': params,json = kw,None
    else: params,json = None,kw
    if use_user:
        if not self.user_token: raise ValueError("User token required for this operation")
        kw['headers'] = {'Authorization': self.user_token}
    r = await self.cli.request(method, path, data=data, files=files, params=params, json=json)
    if r.status_code == 429:
        await asyncio.sleep(float(r.headers.get('Retry-After', 1))+0.1)
        return await self._req(method, path, use_user=use_user, **kw)
    if r.status_code >= 400:
        try: d = r.json()
        except Exception: d = {}
        raise DiscordError(d.get('code', 0), d.get('message', r.text[:200]), r.status_code)
    if r.status_code == 204: return 'done'
    try: return r.json()
    except Exception as e: raise Exception(f"Failed to parse response: {r.text}; {e}")

Discord's REST endpoints return JSON with many fields. Rather than defining properties for every field, we use a flexible base class pattern:

`DiscordObject` provides `__getitem__` and `__getattr__` so you can access data as `obj.name` or `obj['name']`. The `__dir__` method enables autocomplete in notebooks and solveit dialogs—just type `obj.` and see all available fields!

This means when Discord adds new fields to their API, our code automatically supports them without changes.

In [ ]:
#| export
class DiscordObject(AttrDict):
    def __init__(self, data, client):
        super().__init__(data)
        self._client = getattr(client, '_client', client)
    
    async def __call__(self, meth, path, **kw): return await self._client._req(meth, path, **kw)
    def obj(self, nm, d): return globals()[nm](d, self)
    def coll(self, nm, ds): return globals()[nm+'s'](self.obj(nm,d) for d in ds)

Discord's hierarchy is **Guild → Channels → Messages**. A Guild (server) contains channels, and channels contain messages. Each has its own REST endpoints:
- `GET /guilds/{id}` - fetch guild info
- `GET /guilds/{id}/channels` - list channels
- `GET /channels/{id}/messages` - fetch messages
- `POST /channels/{id}/messages` - send a message

We build wrapper classes for each, inheriting from `DiscordObject` and just adding a nice `__repr__`.

In [ ]:
#| export
class Guild(DiscordObject):
    def __repr__(self): return f"Guild(id={self.id}, name={self.name!r})"

We use `@patch` from fastcore to add methods incrementally—great for interactive development where you want to test each piece as you build it. This `guild` method fetches guild data and wraps it in our `Guild` class.

In [ ]:
#| export
@patch
async def guild(self:DiscordClient, guild_id):
    return Guild(await self._req('GET', f'/guilds/{guild_id}'), self)

In [ ]:
# gid = '1493461895615873044'
gid = '1327046393453613076'
gld = await dc.guild(gid); gld

```python
Guild(id=1327046393453613076, name="natedog's server")
```

Channels have a `type` field: 0=text, 2=voice, 4=category. The `Channels` class inherits from `list` and adds `_repr_html_` for nice table display in notebooks/solveit. This pattern—a wrapper class plus a collection class with HTML repr—makes exploring the API much more pleasant.

In [ ]:
#| export
def html_table(items, hdrs, fn):
    if not items: return ""
    rows = (Tr(map(Td, fn(o))) for o in items)
    return to_xml(Table(Thead(Tr(map(Th, hdrs))), Tbody(rows), cls="prose"))

In [ ]:
#| export
class Channel(DiscordObject):
    @property
    async def guild(self):
        if not hasattr(self, '_gld'): self._gld = await self._client.guild(self.guild_id)
        return self._gld
    def __repr__(self): return f"Channel(id={self.id}, name={self.name!r}, type={self.type})"

class Channels(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Type"), lambda c: (c.id, c.name, c.type))

In [ ]:
#| export
@patch
async def channels(self:Guild, limit=None):
    data = await self('GET', f'/guilds/{self.id}/channels')
    if limit: data = data[:limit]
    return self.coll('Channel', data)

In [ ]:
chs = await gld.channels(5)
chs

ID,Name,Type
1327046393453613077,Text Channels,4
1327046393453613078,Voice Channels,4
1327046393453613079,general,0
1327046393453613080,General,2
1327954661960978512,private,0


In [ ]:
#| export
@patch
async def channel(self:DiscordClient, channel_id):
    return Channel(await self._req('GET', f'/channels/{channel_id}'), self)

In [ ]:
chid = '1327046393453613079' # '1493461896139903028'

In [ ]:
ch = await dc.channel(chid)
ch

```python
Channel(id=1327046393453613079, name='general', type=0)
```

Messages are the core of most bot functionality. Note that Discord returns messages in reverse chronological order (newest first), so we `reverse()` the data to get chronological order. The table shows a preview of content, author, and timestamp.

Note: To read message content, your bot needs the `MESSAGE_CONTENT` privileged intent enabled in the Developer Portal! Without it, the `content` field will be empty for messages not sent by your bot or mentioning it.

In [ ]:
#| export
class Message(DiscordObject):
    def __init__(self, data, dobj):
        super().__init__(data, dobj)
        self.raw_content = self.content
        mentions = {m['id']: m['username'] for m in data.get('mentions', [])}
        def repl(m): return f"@{mentions.get(m.group(1), 'unknown')}"
        self['content'] = re.sub(r'<@!?(\d+)>', repl, self.content)

    def __repr__(self): 
        author = self.author['username']
        preview = self.content[:30] + '…' if len(self.content) > 30 else self.content
        return f"Message(id={self.id}, author={author!r}, content={preview!r})"

    @property
    async def channel(self):
        if not hasattr(self, '_ch'): self._ch = Channel(await self('GET', f'/channels/{self.channel_id}'), self._client)
        return self._ch

In [ ]:
#| export
class Messages(list):
    def _repr_html_(self):
        return html_table(self, ("ID", "Author", "Content", "Date"),
            lambda m: (m.id, m.author['username'], m.content[:50], m.timestamp[:10]))

In [ ]:
#| export
@patch
async def messages(self:Channel, limit=50):
    data = await self('GET', f'/channels/{self.id}/messages', limit=limit)
    return self.coll('Message', reversed(data))

In [ ]:
msgs = await ch.messages(5); msgs

ID,Author,Content,Date
1494381457928618014,DBuddy,I'm replying to myself 🤓,2026-04-16
1494381464177999893,DBuddy,Here is a file!,2026-04-16
1494381474240270511,DBuddy,Test our event listener! Otters are awesome 🦦,2026-04-16
1494381547586064515,DBuddy,"Houston, do we have a problem?",2026-04-16
1494381604842504372,DBuddy,Did we re-identify?,2026-04-16


Guild search uses snowflake IDs for date filtering (`min_id`/`max_id`), but we autoconvert `'YYYY-MM-DD'` strings for convenience for `before/after`.

In [ ]:
#| export
depoch = 1420070400000
def date2snowflake(date_str):
    "Convert 'YYYY-MM-DD' to a Discord snowflake ID"
    dt = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)
    return str(int((dt.timestamp() * 1000 - depoch) * (2**22)))

@patch
async def search(self:Guild, content=None, author_id=None, channel_id=None, mentions=None,
                 has=None, before=None, after=None, pinned=None, sort_by=None, sort_order=None,
                 offset=None, limit=None, use_user=False):
    "Search guild messages. `before`/`after` accept 'YYYY-MM-DD' strings or snowflake IDs."
    if before and not str(before).isdigit(): before = date2snowflake(before)
    if after and not str(after).isdigit(): after = date2snowflake(after)
    r = await self('GET', f'/guilds/{self.id}/messages/search', use_user=use_user,
        content=content, author_id=author_id, channel_id=channel_id,
        mentions=mentions, has=has, min_id=after, max_id=before, pinned=pinned,
        sort_by=sort_by, sort_order=sort_order, offset=offset, limit=limit)
    return self.coll('Message', [m[0] for m in r['messages']])

In [ ]:
await gld.search(after='2026-02-16', limit=5)

ID,Author,Content,Date
1494381604842504372,DBuddy,Did we re-identify?,2026-04-16
1494381547586064515,DBuddy,"Houston, do we have a problem?",2026-04-16
1494381474240270511,DBuddy,Test our event listener! Otters are awesome 🦦,2026-04-16
1494381464177999893,DBuddy,Here is a file!,2026-04-16
1494381457928618014,DBuddy,I'm replying to myself 🤓,2026-04-16


Sometimes you need to search by name rather than snowflake ID. `find_member` searches the guild's members by username, nickname, or display name using Discord's member search endpoint, and returns the first match's user ID. This makes it easy to chain into `search`.

In [ ]:
#| export
@patch
async def find_member(self:Guild, name):
    "Search guild members by name/nick, return first match's user ID or None"
    data = await self( 'GET', f'/guilds/{self.id}/members/search', query=name, limit=1)
    return data[0]['user']['id'] if data else None

In [ ]:
uid = await gld.find_member('jeremyhoward')
assert uid
await gld.search(author_id=uid, limit=5)

AssertionError: 

The `members` property fetches all guild members and returns their display names, preferring server nickname over global display name over username. Note: this requires the **Server Members Intent** enabled in the Developer Portal (Bot → Privileged Gateway Intents).

In [ ]:
#| export
@patch(as_prop=True)
async def members(self:Guild):
    "List all guild members' display names (nick > global_name > username)"
    data = await self('GET', f'/guilds/{self.id}/members', limit=1000)
    return [m.get('nick') or m['user'].get('global_name') or m['user']['username'] for m in data]

In [ ]:
(await gld.members)[:5]

['nathan', 'SearchBuddy', 'DBuddy', 'Dizcord Util Bot', 'Search Agent']

Sending messages is a POST request with JSON body. Pass `reply_id` to thread a reply under an existing message. For file attachments, we switch to `multipart/form-data`—Discord expects a `payload_json` field with the message JSON, plus `files[n]` fields for each file.

In [ ]:
#| export
@patch
async def send(self:Channel, content='', files=None, reply_id=None):
    "Send a message with optional file attachments"
    path = f'/channels/{self.id}/messages'
    mref = {'message_id': reply_id} if reply_id else None
    payload = filter_values(dict(content=content, message_reference=mref), negate(is_(None)))
    fs = [('files[%d]' % i, (f.name, f.read_bytes(), None)) for i,f in enumerate(map(Path, listify(files)))]
    r = await self('POST', path, data={'payload_json': json.dumps(payload)}, files=fs)
    return Message(r, self)

In [ ]:
msg = await ch.send('Hi, from Solveit!'); msg

```python
Message(id=1494383523015164087, author='DBuddy', content='Hi, from Solveit!')
```

In [ ]:
reply_msg = await ch.send("I'm replying to myself 🤓", reply_id=msg.id); reply_msg

```python
Message(id=1494383523946303661, author='DBuddy', content="I'm replying to myself 🤓")
```

In [ ]:
await msg.channel

```python
Channel(id=1327046393453613079, name='general', type=0)
```

In [ ]:
msg = await ch.send('Here is a file!', files=['../README.md']); msg

```python
Message(id=1494383525539872970, author='DBuddy', content='Here is a file!')
```

In [ ]:
#| export
@patch
@delegates(Guild.search, but=['channel_id'])
async def search(self:Channel, **kwargs): return await (await self.guild).search(channel_id=self.id, **kwargs)

In [ ]:
await ch.search(has='file', limit=1)

ID,Author,Content,Date
1494381464177999893,DBuddy,Here is a file!,2026-04-16


Discord messages can have file attachments. The `Attachment` class wraps them as `DiscordObject`s—giving you attribute access to `filename`, `size`, `content_type`, `url`, etc. The `fetch` method downloads the file content using the existing httpx client.

In [ ]:
#| export
class Attachment(DiscordObject):
    async def fetch(self): return (await self._client.cli.get(self.url)).content
    def __repr__(self): return f"Attachment(filename={self.filename!r}, size={self.size}, type={self.content_type})"
    def _repr_html_(self):
        if self.content_type.startswith('image/'):
            return f'<img src="{self.url}" style="max-width:400px"><br><small>{self.filename} ({self.size:,}B)</small>'
        return f'<code>{self.filename}</code> ({self.content_type}, {self.size:,}B)'

@patch(as_prop=True)
def attachments(self:Message): return [Attachment(a, self._client) for a in self.get('attachments', [])]

In [ ]:
atts = msg.attachments; atts

[Attachment(filename='README.md', size=28163, type=text/markdown; charset=utf-8)]

In [ ]:
readme = (await atts[0].fetch()).decode()
print(readme[:16])

# cordslite 🍺





DMs (Direct Messages) are just regular channels in Discord's API — no special handling needed! To start a DM conversation, POST to `/users/@me/channels` with a `recipient_id`. Discord returns a standard channel object, so `send()` and `messages()` work exactly as they do for guild channels.

To detect DMs in the gateway, check for the absence of `guild_id` — DM messages don't belong to any guild, so this field is missing or `None`. This makes it easy to route DM vs guild messages in your bot's handler.

In [ ]:
#| export
@patch
async def create_dm(self:DiscordClient, user_id):
    r = await self._req('POST', '/users/@me/channels', recipient_id=user_id)
    return Channel(r, self)

In [ ]:
# # Commented out so we don't spam Nate
# dm = await dc.create_dm('346450717025894400')  # nathan's user ID
# await dm.send('Hello from DMs!')

Members vs Users: A **User** is a global Discord account. A **Member** is a user *within a specific guild*—it has guild-specific data like nickname, roles, and join date. The `nick or user['username']` pattern shows the server nickname if set, otherwise falls back to the global username.

Note: The members endpoint requires the `GUILD_MEMBERS` privileged intent enabled in your bot settings on the Developer Portal!

In [ ]:
#| export
class User(DiscordObject):
    def __repr__(self): return f"User(id={self.id}, username={self.username!r})"

class Member(DiscordObject):
    def __repr__(self):
        name = self.nick or self.user['username']
        return f"Member(id={self.user['id']}, name={name!r}, joined={self.joined_at[:10]})"

class Members(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Joined", "Roles"), lambda m: (m.user['id'], m.nick or m.user['username'], m.joined_at[:10], len(m.roles)))

@patch
async def members(self:Guild, limit=100):
    data = await self('GET', f'/guilds/{self.id}/members', limit=limit)
    return self.coll('Member', data)

In [ ]:
mems = await gld.members(5); mems

ID,Name,Joined,Roles
346450717025894400,nathan,2025-01-09,0
1327047896436178954,SearchBuddy,2025-01-09,1
1361823507679543306,DBuddy,2025-04-15,1
1448038710229733398,Dizcord Util Bot,2025-12-09,1
1467222191182712986,Search Agent,2026-01-31,1


In [ ]:
#| export
@patch
async def search_all(self:Guild, limit=500, delay=1.0, max_age_days=None, show=False, **kwargs):
    "Paginated search returning up to `limit` messages"
    all_msgs, offset = [], 0
    while len(all_msgs) < limit:
        msgs = await self.search(offset=offset, limit=25, **kwargs)
        if not msgs: break
        if max_age_days is not None:
            cutoff = datetime.now(timezone.utc) - timedelta(days=max_age_days)
            msgs = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
        all_msgs.extend(msgs)
        offset += 25
        if show: print(f'Fetched {len(msgs)} more')
        if len(msgs) < 25: break
        await asyncio.sleep(delay)
    return Messages(all_msgs[:limit])

@patch
@delegates(Guild.search_all, but=['channel_id'])
async def search_all(self:Channel, **kwargs):
    gld = await self.guild
    return await gld.search_all(channel_id=self.id, **kwargs)

In [ ]:
# r = await ch.search_all()
# len(r)

In [ ]:
#| export
@patch
async def delete(self:Message):
    "Delete this message"
    return await self('DELETE', f'/channels/{self.channel_id}/messages/{self.id}')

@patch
async def delete_message(self:Channel, message_id):
    "Delete a message by ID"
    return await self('DELETE', f'/channels/{self.id}/messages/{message_id}')

@patch
async def bulk_delete(self:Channel, message_ids):
    "Bulk delete messages (must be <14 days old)"
    return await self('POST', f'/channels/{self.id}/messages/bulk-delete', messages=[str(i) for i in message_ids])

In [ ]:
#| export
@patch
async def create_thread(self:Message, name):
    "Create a thread from this message"
    r = await self('POST', f'/channels/{self.channel_id}/messages/{self.id}/threads', name=name)
    return Channel(r, self)

@patch
async def thread(self:DiscordClient, thread_id):
    "Fetch a thread (which is a Channel)"
    return Channel(await self._req('GET', f'/channels/{thread_id}'), self)

In [ ]:
#| export
@patch
async def _del_rotating(self:Channel, message_ids, delay=0.5, show=False):
    if show: print(f'Deleting {len(message_ids)} messages...')
    for i,mid in enumerate(message_ids):
        use_user = i % 2 == 1 and bool(self._client.user_token)
        await self('DELETE', f'/channels/{self.id}/messages/{mid}', use_user=use_user)
        await asyncio.sleep(delay)
        if show: print('.', end='', flush=True)
    return len(message_ids)

@patch
async def search_and_delete_all(self:Channel, content, delay=2, show=False, **kwargs):
    "Bulk delete recent msgs, individually delete older ones"
    assert content, "Must include content to search for"
    cutoff = datetime.now(timezone.utc) - timedelta(days=13)
    gld = await self.guild
    msgs = await gld.search_all(content=content, channel_id=self.id, show=show, **kwargs)
    recent = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
    old = [m for m in msgs if datetime.fromisoformat(m.timestamp) <= cutoff]
    for i in range(0, len(recent), 20):
        ms = recent[i:i+20]
        await self.bulk_delete([m.id for m in ms])
        if show: print(f'Bulk deleted {len(ms)}')
        await asyncio.sleep(delay)
    if old: await self._del_rotating([m.id for m in old], show=show, delay=delay)
    return len(msgs)

## Gateway API

The Gateway is Discord's WebSocket connection for **real-time events**. Unlike the REST API (request/response), the Gateway pushes events to you as they happen—new messages, reactions, user joins, etc.

The connection lifecycle is:
1. Fetch WSS URL from `/gateway/bot`
2. Connect to WebSocket
3. Receive **Hello** event with heartbeat interval
4. Start heartbeat loop to keep connection alive
5. Send **Identify** with your token and intents
6. Receive **Ready** event—you're connected!

**Intents** tell Discord which events you want. They're bitwise flags you OR together:
- `1 << 0` = GUILDS (guild/channel events)
- `1 << 9` = GUILD_MESSAGES (message events)
- `1 << 15` = MESSAGE_CONTENT (privileged—see message content)

The Gateway API lets apps open secure WebSocket connections with Discord to receive events about actions that take place in a server/guild, like when a channel is updated or a role is created. There are a few cases where apps will _also_ use Gateway connections to update or request resources, like when updating voice state.

In _most_ cases, performing REST operations on Discord resources can be done using the [HTTP API](https://docs.discord.com/developers/reference#http-api) rather than the Gateway API.

The Gateway is Discord’s form of real-time communication used by clients (including apps), so there are nuances and data passed that simply isn’t relevant to apps. Interacting with the Gateway can be tricky, but there are [community-built libraries](https://docs.discord.com/developers/developer-tools/community-resources#libraries) with built-in support that simplify the most complicated bits and pieces. If you’re planning on writing a custom implementation, be sure to read the following documentation in its entirety so you understand the sacred secrets of the Gateway (or at least those that matter for apps).

## 

[​

](https://docs.discord.com/developers/events/gateway#gateway-events)

Gateway Events

Gateway events are [payloads](https://docs.discord.com/developers/events/gateway-events#payload-structure) sent over a [Gateway connection](https://docs.discord.com/developers/events/gateway#connections)—either from an app to Discord, or from Discord to an app. An app typically [_sends_ events](https://docs.discord.com/developers/events/gateway#sending-events) when connecting and managing its connection to the Gateway, and [_receives_ events](https://docs.discord.com/developers/events/gateway#receiving-events) when listening to actions taking place in a server. All Gateway events are encapsulated in a [Gateway event payload](https://docs.discord.com/developers/events/gateway-events#payload-structure). A full list of Gateway events and their details are in the [Gateway events documentation](https://docs.discord.com/developers/events/gateway-events).

###### Example Gateway Event

```
{
  "op": 0,
  "d": {},
  "s": 42,
  "t": "GATEWAY_EVENT_NAME"
}
```

Details about Gateway event payloads are in the [Gateway events documentation](https://docs.discord.com/developers/events/gateway-events#payload-structure).

### 

[​

](https://docs.discord.com/developers/events/gateway#sending-events)

Sending Events

When sending a Gateway event (like when [performing an initial handshake](https://docs.discord.com/developers/events/gateway-events#identify) or [updating presence](https://docs.discord.com/developers/events/gateway-events#update-presence)), your app must send an [event payload object](https://docs.discord.com/developers/events/gateway-events#payload-structure) with a valid opcode (`op`) and inner data object (`d`).

Specific rate limits are applied when sending events, which you can read about in the [Rate Limiting](https://docs.discord.com/developers/events/gateway#rate-limiting) section.

Event payloads sent over a Gateway connection:

1.  Must be serialized in [plain-text JSON or binary ETF](https://docs.discord.com/developers/events/gateway#encoding-and-compression).
2.  Must not exceed 4096 bytes. If an event payload _does_ exceed 4096 bytes, the connection will be closed with a [`4002` close event code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes).

All events that your app can send via a connection are in [Gateway event documentation](https://docs.discord.com/developers/events/gateway-events#send-events).

### 

[​

](https://docs.discord.com/developers/events/gateway#receiving-events)

Receiving Events

Receiving a Gateway event from Discord (like when [a reaction is added to a message](https://docs.discord.com/developers/events/gateway-events#message-reaction-add)) is much more common (and slightly more complex) than sending them. While some events are sent to your app automatically, most events require your app to define intents when [Identifying](https://docs.discord.com/developers/events/gateway#identifying). Intents are bitwise values that can be ORed (`|`) to indicate which events (or groups of events) you want Discord to send your app. A list of intents and their corresponding events are listed in the [intents section](https://docs.discord.com/developers/events/gateway#gateway-intents). When receiving events, you can also configure _how_ events will be sent to your app, like the [encoding and compression](https://docs.discord.com/developers/events/gateway#encoding-and-compression), or whether [sharding should be enabled](https://docs.discord.com/developers/events/gateway#sharding)). All events that your app can receive via a connection are in the [Gateway event documentation](https://docs.discord.com/developers/events/gateway-events#receive-events).

#### 

[​

](https://docs.discord.com/developers/events/gateway#dispatch-events)

Dispatch Events

[Dispatch (opcode `0`)](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-opcodes) events are the most common type of event your app will receive. _Most_ Gateway events which represent actions taking place in a guild will be sent to your app as Dispatch events. When your app is parsing a Dispatch event:

-   The `t` field can be used to determine which [Gateway event](https://docs.discord.com/developers/events/gateway-events#receive-events) the payload represents the data you can expect in the `d` field.
-   The `s` field represents the sequence number of the event, which is the relative order in which it occurred. You need to cache the most recent non-null `s` value for heartbeats, and to pass when [Resuming](https://docs.discord.com/developers/events/gateway#resuming) a connection.

## 

[​

](https://docs.discord.com/developers/events/gateway#connections)

Connections

Gateway connections are persistent WebSockets which introduce more complexity than sending HTTP requests or responding to interactions (like Slash Commands). When interacting with the Gateway, your app must know how to open the initial connection, as well as maintain it and handle any disconnects.

### 

[​

](https://docs.discord.com/developers/events/gateway#connection-lifecycle)

Connection Lifecycle

There are nuances that aren’t included in the overview below. More details about each step and event can be found in the individual sections below.

At a high-level, Gateway connections consist of the following cycle: ![Flowchart with an overview of Gateway connection lifecycle](https://mintcdn.com/discord/wUt1cakZ575C5Zsi/images/events/gateway-lifecycle.svg?fit=max&auto=format&n=wUt1cakZ575C5Zsi&q=85&s=06d8f878b6eb902f1ce680dcb2837b63#ai)

1.  App establishes a connection with the Gateway after fetching and caching a WSS URL using the [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway) or [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot) endpoint.
2.  Discord sends the app a [Hello (opcode `10`)](https://docs.discord.com/developers/events/gateway#hello-event) event containing a heartbeat interval in milliseconds. **Read the section on [Connecting](https://docs.discord.com/developers/events/gateway#connecting)**
3.  Start the Heartbeat interval. App must send a [Heartbeat (opcode `1`)](https://docs.discord.com/developers/events/gateway-events#heartbeat) event, then continue to send them every heartbeat interval until the connection is closed. **Read the section on [Sending Heartbeats](https://docs.discord.com/developers/events/gateway#sending-heartbeats)**
    -   Discord will respond to each Heartbeat event with a [Heartbeat ACK (opcode `11`)](https://docs.discord.com/developers/events/gateway#sending-heartbeats) event to confirm it was received. If an app doesn’t receive a Heartbeat ACK, it should close the connection and reconnect.
    -   Discord may send the app a [Heartbeat (opcode `1`)](https://docs.discord.com/developers/events/gateway-events#heartbeat) event, in which case the app should send a Heartbeat event immediately.
4.  App sends an [Identify (opcode `2`)](https://docs.discord.com/developers/events/gateway-events#identify) event to perform the initial handshake with the Gateway. **Read the section on [Identifying](https://docs.discord.com/developers/events/gateway#identifying)**
5.  Discord sends the app a [Ready (opcode `0`)](https://docs.discord.com/developers/events/gateway-events#ready) event which indicates the handshake was successful and the connection is established. The Ready event contains a `resume_gateway_url` that the app should keep track of to determine the WebSocket URL an app should use to Resume. **Read the section on [the Ready event](https://docs.discord.com/developers/events/gateway#ready-event)**
6.  The connection may be dropped for a variety of reasons at any time. Whether the app can [Resume](https://docs.discord.com/developers/events/gateway#resuming) the connection or whether it must re-identify is determined by a variety of factors like the [opcode](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-opcodes) and [close code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes) that it receives. **Read the section on [Disconnecting](https://docs.discord.com/developers/events/gateway#disconnecting)**
7.  If an app **can** resume/reconnect, it should open a new connection using `resume_gateway_url` with the same version and encoding, then send a [Resume (opcode `6`)](https://docs.discord.com/developers/events/gateway-events#resume) event. If an app **cannot** resume/reconnect, it should open a new connection using the cached URL from step #1, then repeat the whole Gateway cycle. _Yipee!_ **Read the section on [Resuming](https://docs.discord.com/developers/events/gateway#resuming)**

### 

[​

](https://docs.discord.com/developers/events/gateway#connecting)

Connecting

Before your app can establish a connection to the Gateway, it should call the [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway) or the [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot) endpoint. Either endpoint will return a payload with a `url` field whose value is the WSS URL you can use to open a WebSocket connection. In addition to the URL, [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot) contains additional information about the recommended number of shards and the session start limits for your app. When initially calling either [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway) or [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot), you should cache the value of the `url` field and use that when re-connecting to the Gateway. When connecting to the URL, it’s a good idea to explicitly pass the API version and [encoding](https://docs.discord.com/developers/events/gateway#encoding-and-compression) as query parameters. You can also optionally include whether Discord should [compress](https://docs.discord.com/developers/events/gateway#encoding-and-compression) data that it sends your app.

`wss://gateway.discord.gg/?v=10&encoding=json` is an example of a WSS URL an app may use to connect to the Gateway.

###### Gateway URL Query String Params

| Field | Type | Description | Accepted Values |
| --- | --- | --- | --- |
| v | integer | [API Version](https://docs.discord.com/developers/reference#api-versioning) to use | [API version](https://docs.discord.com/developers/reference#api-versioning-api-versions) |
| encoding | string | The [encoding](https://docs.discord.com/developers/events/gateway#encoding-and-compression) of received gateway packets | `json` or `etf` |
| compress? | string | The optional [transport compression](https://docs.discord.com/developers/events/gateway#transport-compression) of gateway packets | `zlib-stream` or `zstd-stream` |

#### 

[​

](https://docs.discord.com/developers/events/gateway#hello-event)

Hello Event

Once connected to the Gateway, your app will receive a [Hello (opcode `10`)](https://docs.discord.com/developers/events/gateway#hello-event) event that contains your connection’s heartbeat interval (`heartbeat_interval`). The heartbeat interval indicates a length of time in milliseconds that you should use to determine how often your app needs to send a Heartbeat event in order to maintain the active connection. Heartbeating is detailed in the [Sending Heartbeats](https://docs.discord.com/developers/events/gateway#sending-heartbeats) section.

###### Example Hello Event

```
{
  "op": 10,
  "d": {
    "heartbeat_interval": 45000
  }
}
```

### 

[​

](https://docs.discord.com/developers/events/gateway#sending-heartbeats)

Sending Heartbeats

Heartbeats are pings used to let Discord know that your app is still actively using a Gateway connection. After connecting to the Gateway, your app should send heartbeats (as described below) in a background process until the Gateway connection is closed.

#### 

[​

](https://docs.discord.com/developers/events/gateway#heartbeat-interval)

Heartbeat Interval

When your app opens a Gateway connection, it will receive a [Hello (opcode `10`)](https://docs.discord.com/developers/events/gateway#hello-event) event which includes a `heartbeat_interval` field that has a value representing a length of time in milliseconds. Upon receiving the Hello event, your app should wait `heartbeat_interval * jitter` where `jitter` is any random value between 0 and 1, then send its first [Heartbeat (opcode `1`)](https://docs.discord.com/developers/events/gateway-events#heartbeat) event. From that point until the connection is closed, your app must continually send Discord a heartbeat every `heartbeat_interval` milliseconds. If your app fails to send a heartbeat event in time, your connection will be closed and you will be forced to [Resume](https://docs.discord.com/developers/events/gateway#resuming). When sending a heartbeat, your app will need to include the last sequence number your app received in the `d` field. The sequence number is sent to your app in the [event payload](https://docs.discord.com/developers/events/gateway-events#payload-structure) in the `s` field. If your app hasn’t received any events yet, you can just pass `null` in the `d` field.

In the first heartbeat, `jitter` is an offset value between 0 and `heartbeat_interval` that is meant to prevent too many clients (both desktop and apps) from reconnecting their sessions at the exact same time (which could cause an influx of traffic).

You _can_ send heartbeats before the `heartbeat_interval` elapses, but you should avoid doing so unless necessary. There is already tolerance in the `heartbeat_interval` that will cover network latency, so you don’t need to account for it in your implementation. When you send a Heartbeat event, Discord will respond with a [Heartbeat ACK (opcode `11`)](https://docs.discord.com/developers/events/gateway#heartbeat-interval-example-heartbeat-ack) event, which is an acknowledgement that the heartbeat was received:

###### Example Heartbeat ACK

```
{
  "op": 11
}
```

In the event of a service outage where you stay connected to the Gateway, you should continue to send heartbeats and receive heartbeat ACKs. The Gateway will eventually respond and issue a session once it’s able to.

If a client does not receive a heartbeat ACK between its attempts at sending heartbeats, this may be due to a failed or “zombied” connection. The client should immediately terminate the connection with any close code besides `1000` or `1001`, then reconnect and attempt to [Resume](https://docs.discord.com/developers/events/gateway#resuming).

#### 

[​

](https://docs.discord.com/developers/events/gateway#heartbeat-requests)

Heartbeat Requests

In addition to the Heartbeat interval, Discord may request additional heartbeats from your app by sending a [Heartbeat (opcode `1`)](https://docs.discord.com/developers/events/gateway-events#heartbeat) event. Upon receiving the event, your app should immediately send back another Heartbeat event without waiting the remainder of the current interval. Just like with the interval, Discord will respond with an [Heartbeat ACK (opcode `11`)](https://docs.discord.com/developers/events/gateway#heartbeat-interval-example-heartbeat-ack) event.

### 

[​

](https://docs.discord.com/developers/events/gateway#identifying)

Identifying

After the connection is open and your app is sending heartbeats, you should send an [Identify (opcode `2`)](https://docs.discord.com/developers/events/gateway-events#identify) event. The Identify event is an initial handshake with the Gateway that’s required before your app can begin sending or receiving most Gateway events. Apps are limited by maximum concurrency (`max_concurrency` in the [session start limit object](https://docs.discord.com/developers/events/gateway#session-start-limit-object)) when identifying. If your app exceeds this limit, Discord will respond with a [Invalid Session (opcode `9`)](https://docs.discord.com/developers/events/gateway-events#invalid-session) event. After your app sends a valid Identify payload, Discord will respond with a [Ready](https://docs.discord.com/developers/events/gateway-events#ready) event which indicates that your app is in a successfully-connected state with the Gateway. The Ready event is sent as a standard [Dispatch (opcode `0`)](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-opcodes).

Clients are limited to 1000 `IDENTIFY` calls to the websocket in a 24-hour period. This limit is global and across all shards, but does not include `RESUME` calls. Upon hitting this limit, all active sessions for the app will be terminated, the bot token will be reset, and the owner will receive an email notification. It’s up to the owner to update their application with the new token.

###### Example Identify Payload

Below is a minimal `IDENTIFY` payload. `IDENTIFY` supports additional fields for other session properties like payload compression and an initial presence state. See the [Identify Structure](https://docs.discord.com/developers/events/gateway-events#identify-identify-structure) for details about the event.

```
{
  "op": 2,
  "d": {
    "token": "my_token",
    "intents": 513,
    "properties": {
      "os": "linux",
      "browser": "my_library",
      "device": "my_library"
    }
  }
}
```

#### 

[​

](https://docs.discord.com/developers/events/gateway#ready-event)

Ready event

As mentioned above, the [Ready](https://docs.discord.com/developers/events/gateway-events#ready) event is sent to your app after it sends a valid Identify payload. The Ready event includes state, like the guilds your app is in, that it needs to start interacting with the rest of the platform. The Ready event also includes fields that you’ll need to cache in order to eventually [Resume](https://docs.discord.com/developers/events/gateway#resuming) your connection after disconnects. Two fields in particular are important to call out:

-   `resume_gateway_url` is a WebSocket URL that your app should use when it Resumes after a disconnect. The `resume_gateway_url` should be used instead of the URL [used when connecting](https://docs.discord.com/developers/events/gateway#connecting).
-   `session_id` is the ID for the Gateway session for the new connection. It’s required to know which stream of events were associated with your disconnection connection.

Full details about the Ready event is in the [Gateway events documentation](https://docs.discord.com/developers/events/gateway-events).

### 

[​

](https://docs.discord.com/developers/events/gateway#disconnecting)

Disconnecting

Gateway disconnects happen for a variety of reasons, and may be initiated by Discord or by your app.

#### 

[​

](https://docs.discord.com/developers/events/gateway#handling-a-disconnect)

Handling a Disconnect

Due to Discord’s architecture, disconnects are a semi-regular event and should be expected and handled. When your app encounters a disconnect, it will typically be sent a [close code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes) which can be used to determine whether you can reconnect and [Resume](https://docs.discord.com/developers/events/gateway#resuming) the session, or whether you have to start over and re-Identify. After you determine whether or not your app can reconnect, you will do one of the following:

-   If you determine that your app _can_ reconnect and resume the previous session, then you should reconnect using the `resume_gateway_url` and `session_id` from the [Ready](https://docs.discord.com/developers/events/gateway-events#ready) event. Details about when and how to resume can be found in the [Resuming](https://docs.discord.com/developers/events/gateway#resuming) section.
-   If you _cannot_ reconnect **or the reconnect fails**, you should open a new connection using the URL from the initial call to [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway) or [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot). In the case you cannot reconnect, you’ll have to re-identify after opening a new connection.

A full list of the close codes can be found in the [Response Codes](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes) documentation.

#### 

[​

](https://docs.discord.com/developers/events/gateway#initiating-a-disconnect)

Initiating a Disconnect

When you close the connection to the gateway with close code `1000` or `1001`, your session will be invalidated and your bot will appear offline. If you simply close the TCP connection or use a different close code, the session will remain active and timeout after a few minutes. This can be useful when you’re [Resuming](https://docs.discord.com/developers/events/gateway#resuming) the previous session.

### 

[​

](https://docs.discord.com/developers/events/gateway#resuming)

Resuming

When your app is disconnected, Discord has a process for reconnecting and resuming, which allows your app to replay any lost events starting from the last sequence number it received. After Resuming, your app will receive the missed events in the same way it would have had the connection had stayed active. Unlike the initial connection, your app does **not** need to re-Identify when Resuming. There are a handful of scenarios when your app should attempt to resume:

1.  It receives a [Reconnect (opcode `7`)](https://docs.discord.com/developers/events/gateway-events#reconnect) event
2.  It’s disconnected with a close code that indicates it can reconnect. A list of close codes is in the [Opcodes and Status Codes](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes) documentation.
3.  It’s disconnected but doesn’t receive _any_ close code.
4.  It receives an [Invalid Session (opcode `9`)](https://docs.discord.com/developers/events/gateway-events#invalid-session) event with the `d` field set to `true`. This is an unlikely scenario, but it is possible.

#### 

[​

](https://docs.discord.com/developers/events/gateway#preparing-to-resume)

Preparing to Resume

Before your app can send a [Resume (opcode `6`)](https://docs.discord.com/developers/events/gateway-events#resume) event, it will need three values: the `session_id` and the `resume_gateway_url` from the [Ready](https://docs.discord.com/developers/events/gateway#ready-event) event, and the sequence number (`s`) from the last Dispatch (opcode `0`) event it received before the disconnect. After the connection is closed, your app should open a new connection using `resume_gateway_url` rather than the URL used to initially connect, with the same query parameters from the initial [Connection](https://docs.discord.com/developers/events/gateway#connecting). If your app doesn’t use the `resume_gateway_url` when reconnecting, it will experience disconnects at a higher rate than normal. Once the new connection is opened, your app should send a [Gateway Resume](https://docs.discord.com/developers/events/gateway-events#resume) event using the `session_id` and sequence number mentioned above. When sending the event, `session_id` will have the same field name, but the last sequence number will be passed as `seq` in the data object (`d`). When Resuming, you do not need to send an Identify event after opening the connection. If successful, the Gateway will send the missed events in order, finishing with a [Resumed](https://docs.discord.com/developers/events/gateway-events#resumed) event to signal event replay has finished and that all subsequent events will be new.

When resuming with the `resume_gateway_url` you need to provide the same version and encoding as the initial connection.

It’s possible your app won’t reconnect in time to Resume, in which case it will receive an [Invalid Session (opcode `9`)](https://docs.discord.com/developers/events/gateway-events#invalid-session) event. If the `d` field is set to `false` (which is most of the time), your app should disconnect. After disconnect, your app should create a new connection with your cached URL from the [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway) or the [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot) endpoint, then send an [Identify (opcode `2`)](https://docs.discord.com/developers/events/gateway-events#identify) event.

###### Example Gateway Resume Event

```
{
  "op": 6,
  "d": {
    "token": "my_token",
    "session_id": "session_id_i_stored",
    "seq": 1337
  }
}
```

## 

[​

](https://docs.discord.com/developers/events/gateway#gateway-intents)

Gateway Intents

Maintaining a stateful application can be difficult when it comes to the amount of data your app is expected to process over a Gateway connection, especially at scale. Gateway intents are a system to help you lower the computational burden. Intents are bitwise values passed in the `intents` parameter when [Identifying](https://docs.discord.com/developers/events/gateway#identifying) which correlate to a set of related events. For example, the event sent when a guild is created (`GUILD_CREATE`) and when a channel is updated (`CHANNEL_UPDATE`) both require the same `GUILDS (1 << 0)` intent (as listed in the table below). If you do not specify an intent when identifying, you will not receive _any_ of the Gateway events associated with that intent.

Intents are optionally supported on the v6 gateway but required as of v8

Two types of intents exist:

-   **Standard intents** can be passed by default. You don’t need any additional permissions or configurations.
-   [**Privileged intents**](https://docs.discord.com/developers/events/gateway#privileged-intents) require you to toggle the intent for your app in your app’s settings within the Developer Portal before passing said intent. For verified apps (required for apps in 100+ guilds), the intent must also be approved after the verification process to use the intent. More information about privileged intents can be found [in the section below](https://docs.discord.com/developers/events/gateway#privileged-intents).

The connection with your app will be closed if it passes invalid intents ([`4013` close code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes)), or a privileged intent that hasn’t been configured or approved for your app ([`4014` close code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes)).

### 

[​

](https://docs.discord.com/developers/events/gateway#list-of-intents)

List of Intents

Below is a list of all intents and the Gateway events associated with them. Any events _not_ listed means it’s not associated with an intent and will always be sent to your app. All events, including those that aren’t associated with an intent, are in the [Gateway events](https://docs.discord.com/developers/events/gateway-events) documentation.

```
GUILDS (1 << 0)
  - GUILD_CREATE
  - GUILD_UPDATE
  - GUILD_DELETE
  - GUILD_ROLE_CREATE
  - GUILD_ROLE_UPDATE
  - GUILD_ROLE_DELETE
  - CHANNEL_CREATE
  - CHANNEL_UPDATE
  - CHANNEL_DELETE
  - CHANNEL_PINS_UPDATE
  - THREAD_CREATE
  - THREAD_UPDATE
  - THREAD_DELETE
  - THREAD_LIST_SYNC
  - THREAD_MEMBER_UPDATE
  - THREAD_MEMBERS_UPDATE *
  - STAGE_INSTANCE_CREATE
  - STAGE_INSTANCE_UPDATE
  - STAGE_INSTANCE_DELETE

GUILD_MEMBERS (1 << 1) **
  - GUILD_MEMBER_ADD
  - GUILD_MEMBER_UPDATE
  - GUILD_MEMBER_REMOVE
  - THREAD_MEMBERS_UPDATE *

GUILD_MODERATION (1 << 2)
  - GUILD_AUDIT_LOG_ENTRY_CREATE
  - GUILD_BAN_ADD
  - GUILD_BAN_REMOVE

GUILD_EXPRESSIONS (1 << 3)
  - GUILD_EMOJIS_UPDATE
  - GUILD_STICKERS_UPDATE
  - GUILD_SOUNDBOARD_SOUND_CREATE
  - GUILD_SOUNDBOARD_SOUND_UPDATE
  - GUILD_SOUNDBOARD_SOUND_DELETE
  - GUILD_SOUNDBOARD_SOUNDS_UPDATE

GUILD_INTEGRATIONS (1 << 4)
  - GUILD_INTEGRATIONS_UPDATE
  - INTEGRATION_CREATE
  - INTEGRATION_UPDATE
  - INTEGRATION_DELETE

GUILD_WEBHOOKS (1 << 5)
  - WEBHOOKS_UPDATE

GUILD_INVITES (1 << 6)
  - INVITE_CREATE
  - INVITE_DELETE

GUILD_VOICE_STATES (1 << 7)
  - VOICE_CHANNEL_EFFECT_SEND
  - VOICE_STATE_UPDATE

GUILD_PRESENCES (1 << 8) **
  - PRESENCE_UPDATE

GUILD_MESSAGES (1 << 9)
  - MESSAGE_CREATE
  - MESSAGE_UPDATE
  - MESSAGE_DELETE
  - MESSAGE_DELETE_BULK

GUILD_MESSAGE_REACTIONS (1 << 10)
  - MESSAGE_REACTION_ADD
  - MESSAGE_REACTION_REMOVE
  - MESSAGE_REACTION_REMOVE_ALL
  - MESSAGE_REACTION_REMOVE_EMOJI

GUILD_MESSAGE_TYPING (1 << 11)
  - TYPING_START

DIRECT_MESSAGES (1 << 12)
  - MESSAGE_CREATE
  - MESSAGE_UPDATE
  - MESSAGE_DELETE
  - CHANNEL_PINS_UPDATE

DIRECT_MESSAGE_REACTIONS (1 << 13)
  - MESSAGE_REACTION_ADD
  - MESSAGE_REACTION_REMOVE
  - MESSAGE_REACTION_REMOVE_ALL
  - MESSAGE_REACTION_REMOVE_EMOJI

DIRECT_MESSAGE_TYPING (1 << 14)
  - TYPING_START

MESSAGE_CONTENT (1 << 15) ***

GUILD_SCHEDULED_EVENTS (1 << 16)
  - GUILD_SCHEDULED_EVENT_CREATE
  - GUILD_SCHEDULED_EVENT_UPDATE
  - GUILD_SCHEDULED_EVENT_DELETE
  - GUILD_SCHEDULED_EVENT_USER_ADD
  - GUILD_SCHEDULED_EVENT_USER_REMOVE

AUTO_MODERATION_CONFIGURATION (1 << 20)
  - AUTO_MODERATION_RULE_CREATE
  - AUTO_MODERATION_RULE_UPDATE
  - AUTO_MODERATION_RULE_DELETE

AUTO_MODERATION_EXECUTION (1 << 21)
  - AUTO_MODERATION_ACTION_EXECUTION

GUILD_MESSAGE_POLLS (1 << 24)
  - MESSAGE_POLL_VOTE_ADD
  - MESSAGE_POLL_VOTE_REMOVE

DIRECT_MESSAGE_POLLS (1 << 25)
  - MESSAGE_POLL_VOTE_ADD
  - MESSAGE_POLL_VOTE_REMOVE
```

\* [Thread Members Update](https://docs.discord.com/developers/events/gateway-events#thread-members-update) contains different data depending on which intents are used. \*\* Events under the `GUILD_PRESENCES` and `GUILD_MEMBERS` intents are turned **off by default on all API versions**. If you are using **API v6**, you will receive those events if you are authorized to receive them and have enabled the intents in the Developer Portal. You do not need to use intents on API v6 to receive these events; you just need to enable the flags. If you are using **API v8** or above, intents are mandatory and must be specified when identifying. \*\*\* `MESSAGE_CONTENT` does not represent individual events, but rather affects what data is present for events that could contain message content fields. More information is in the [message content intent](https://docs.discord.com/developers/events/gateway#message-content-intent) section.

### 

[​

](https://docs.discord.com/developers/events/gateway#caveats)

Caveats

[Guild Member Update](https://docs.discord.com/developers/events/gateway-events#guild-member-update) is sent for current-user updates regardless of whether the `GUILD_MEMBERS` intent is set. [Guild Create](https://docs.discord.com/developers/events/gateway-events#guild-create) and [Request Guild Members](https://docs.discord.com/developers/events/gateway-events#request-guild-members) are uniquely affected by intents. See these sections for more information. [Thread Members Update](https://docs.discord.com/developers/events/gateway-events#thread-members-update) by default only includes if the current user was added to or removed from a thread. To receive these updates for other users, request the `GUILD_MEMBERS` [Gateway Intent](https://docs.discord.com/developers/events/gateway#gateway-intents).

### 

[​

](https://docs.discord.com/developers/events/gateway#privileged-intents)

Privileged Intents

Some intents are defined as “privileged” due to the sensitive nature of the data. Currently, those intents include:

-   `GUILD_PRESENCES`
-   `GUILD_MEMBERS`
-   [`MESSAGE_CONTENT`](https://docs.discord.com/developers/events/gateway#message-content-intent)

Apps that qualify for verification **must** be approved for the privileged intent(s) before they can use them. After your app is verified, you can request privileged intents within the app’s settings within the Developer Portal. Before you specify privileged intents in your `IDENTIFY` payload, you must enable the privileged intents your app requires. [Verified apps](https://support-dev.discord.com/hc/en-us/articles/23926564536471-How-Do-I-Get-My-App-Verified) can only use privileged intents _after_ they’ve been approved for them.

Unverified apps can use privileged intents without approval, but still must enable them in their app’s settings. If the app’s verification status changes, it will then have to apply for the privileged intent(s).

In addition to the gateway restrictions described here, Discord’s REST API is also affected by Privileged Intents. For example, to use the [List Guild Members](https://docs.discord.com/developers/resources/guild#list-guild-members) endpoint, you must have the `GUILD_MEMBERS` intent enabled for your application. This behavior is independent of whether the intent is set during `IDENTIFY`.

#### 

[​

](https://docs.discord.com/developers/events/gateway#enabling-privileged-intents)

Enabling Privileged Intents

Before using privileged intents, you must enable them in your app’s settings. In the [Developer Portal](https://discord.com/developers/applications), you can navigate to your app’s settings then toggle the privileged intents on the [**Bot** page](https://discord.com/developers/applications/select/bot) under the “Privileged Gateway Intents” section. You should only toggle privileged intents that your bot _requires to function_. If your app qualifies for [verification](https://support-dev.discord.com/hc/en-us/articles/23926564536471-How-Do-I-Get-My-App-Verified), you must first [verify your app](https://support-dev.discord.com/hc/en-us/articles/23926564536471-How-Do-I-Get-My-App-Verified) and request access to these intents during the verification process. If your app is already verified and you need to request additional privileged intents, you can [contact support](https://dis.gd/support).

#### 

[​

](https://docs.discord.com/developers/events/gateway#gateway-restrictions)

Gateway Restrictions

Privileged intents affect which Gateway events your app is permitted to receive. When using **API v8** and above, all intents (privileged and not) must be specified in the `intents` parameter when Identifying. If you pass a privileged intent in the `intents` parameter without configuring it in your app’s settings, or being approved for it during verification, your Gateway connection will be closed with a ([`4014` close code](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes)).

For **API v6**, you will receive events associated with the privileged intents your app has configured and is authorized to receive _without_ passing those intents into the `intents` parameter when Identifying.

Events associated with the `GUILD_PRESENCES` and `GUILD_MEMBERS` intents are turned off by default regardless of the API version.

#### 

[​

](https://docs.discord.com/developers/events/gateway#http-restrictions)

HTTP Restrictions

In addition to Gateway restrictions, privileged intents also affect the [HTTP API](https://docs.discord.com/developers/reference#http-api) endpoints your app is permitted to call, and the data it can receive. For example, to use the [List Guild Members](https://docs.discord.com/developers/resources/guild#list-guild-members) endpoint, your app must enable the `GUILD_MEMBERS` intent (and be approved for it if eligible for verification). HTTP API restrictions are independent of Gateway restrictions, and are unaffected by which intents your app passes in the `intents` parameter when Identifying.

#### 

[​

](https://docs.discord.com/developers/events/gateway#message-content-intent)

Message Content Intent

`MESSAGE_CONTENT (1 << 15)` is a unique privileged intent that isn’t directly associated with any Gateway events. Instead, access to `MESSAGE_CONTENT` permits your app to receive message content data across the APIs. Any fields affected by the message content intent are noted in the relevant documentation. For example, the `content`, `embeds`, `attachments`, `components`, and `poll` fields in [message objects](https://docs.discord.com/developers/resources/message#message-object) all contain message content and therefore require the intent.

Like other privileged intents, `MESSAGE_CONTENT` must be approved for your app. After your app is verified, you can apply for the intent from your app’s settings within the Developer Portal. You can read more about the message content intent review policy [in the Help Center](https://support-dev.discord.com/hc/en-us/articles/5324827539479).

Apps **without** the intent will receive empty values in fields that contain user-inputted content with a few exceptions:

-   Content in messages that an app sends
-   Content in DMs with the app
-   Content in which the app is [mentioned](https://docs.discord.com/developers/reference#message-formatting-formats)
-   Content of the message a [message context menu command](https://docs.discord.com/developers/interactions/application-commands#message-commands) is used on

## 

[​

](https://docs.discord.com/developers/events/gateway#rate-limiting)

Rate Limiting

This section refers to Gateway rate limits, not [HTTP API rate limits](https://docs.discord.com/developers/topics/rate-limits)

Apps can send 120 [gateway events](https://docs.discord.com/developers/events/gateway-events) per [connection](https://docs.discord.com/developers/events/gateway#connections) every 60 seconds, meaning an average of 2 commands per second. Apps that surpass the limit are immediately disconnected from the Gateway. Similar to other rate limits, repeat offenders will have their API access revoked. Apps also have a limit for [concurrent](https://docs.discord.com/developers/events/gateway#session-start-limit-object) [Identify](https://docs.discord.com/developers/events/gateway#identifying) requests allowed per 5 seconds. If you hit this limit, the Gateway will respond with an [Invalid Session (opcode `9`)](https://docs.discord.com/developers/events/gateway-events#invalid-session).

## 

[​

](https://docs.discord.com/developers/events/gateway#encoding-and-compression)

Encoding and Compression

When [establishing a connection](https://docs.discord.com/developers/events/gateway#connecting) to the Gateway, apps can use the `encoding` parameter to choose whether to communicate with Discord using a plain-text JSON or binary [ETF](https://erlang.org/doc/apps/erts/erl_ext_dist.html) encoding. You can pick whichever encoding type you’re more comfortable with, but both have their own quirks. If you aren’t sure which encoding to use, JSON is generally recommended. Apps can also optionally enable compression to receive zlib-compressed or zstd-compressed packets. [Payload compression](https://docs.discord.com/developers/events/gateway#payload-compression) can only be enabled when using a JSON encoding, but [transport compression](https://docs.discord.com/developers/events/gateway#transport-compression) can be used regardless of encoding type.

### 

[​

](https://docs.discord.com/developers/events/gateway#using-json-encoding)

Using JSON Encoding

When using the plain-text JSON encoding, apps have the option to enable [payload compression](https://docs.discord.com/developers/events/gateway#payload-compression).

#### 

[​

](https://docs.discord.com/developers/events/gateway#payload-compression)

Payload Compression

If an app is using payload compression, it cannot use [transport compression](https://docs.discord.com/developers/events/gateway#transport-compression).

Payload compression enables optional per-packet compression for _some_ events when Discord is sending events over the connection. Payload compression uses the zlib format (see [RFC1950 2.2](https://tools.ietf.org/html/rfc1950#section-2.2)) when sending payloads. To enable payload compression, your app can set `compress` to `true` when sending an [Identify (opcode `2`)](https://docs.discord.com/developers/events/gateway-events#identify) event. Note that even when payload compression is enabled, not all payloads will be compressed. When payload compression is enabled, your app (or library) _must_ detect and decompress these payloads to plain-text JSON before attempting to parse them. If you are using payload compression, the gateway does not implement a shared compression context between events sent. Payload compression will be disabled if you use [transport compression](https://docs.discord.com/developers/events/gateway#transport-compression).

### 

[​

](https://docs.discord.com/developers/events/gateway#using-etf-encoding)

Using ETF Encoding

When using ETF (External Term Format) encoding, there are some specific behaviors you should know:

-   Snowflake IDs are transmitted as 64-bit integers or strings.
-   Your app can’t send compressed messages to the server.
-   When sending payloads, you must use string keys. Using atom keys will result in a [`4002`](https://docs.discord.com/developers/topics/opcodes-and-status-codes#gateway-gateway-close-event-codes) decode error.

See [erlpack](https://github.com/discord/erlpack) for an ETF implementation example.

### 

[​

](https://docs.discord.com/developers/events/gateway#transport-compression)

Transport Compression

Transport compression enables optional compression for all packets when Discord is sending events over the connection. The currently-available transport compression options are `zlib-stream` and `zstd-stream`.

#### 

[​

](https://docs.discord.com/developers/events/gateway#zlib-stream)

zlib-stream

When zlib transport compression is enabled, your app needs to process received data through a single Gateway connection using a shared zlib context. However, each Gateway connection should use its own unique zlib context. When processing transport-compressed data, you should push received data to a buffer until you receive the 4-byte `Z_SYNC_FLUSH` suffix (`00 00 ff ff`). After you receive the `Z_SYNC_FLUSH` suffix, you can then decompress the buffer.

###### Transport Compression Example

```
# Z_SYNC_FLUSH suffix
ZLIB_SUFFIX = b'\x00\x00\xff\xff'
# initialize a buffer to store chunks
buffer = bytearray()
# create a shared zlib inflation context to run chunks through
inflator = zlib.decompressobj()

# ...
def on_websocket_message(msg):
  # always push the message data to your cache
  buffer.extend(msg)

  # check if the last four bytes are equal to ZLIB_SUFFIX
  if len(msg) < 4 or msg[-4:] != ZLIB_SUFFIX:
    return

  # if the message *does* end with ZLIB_SUFFIX,
  # get the full message by decompressing the buffers
  # NOTE: the message is utf-8 encoded.
  msg = inflator.decompress(buffer)
  buffer = bytearray()

  # here you can treat `msg` as either JSON or ETF encoded,
  # depending on your `encoding` param
```

#### 

[​

](https://docs.discord.com/developers/events/gateway#zstd-stream)

zstd-stream

When zstd-stream transport compression is enabled, all data needs to be processed through a zstd decompression context that stays alive for the lifetime of the gateway connection. When processing data, each websocket message corresponds to a single gateway message, but does not end a zstd frame. You will need to repeatedly call ZSTD\_decompressStream until all data in the frame has been processed (ZSTD\_decompressStream will not necessarily return 0, though). Take a look at this [Erlang](https://github.com/silviucpp/ezstd/blob/f3f33b2f6b917f7e8aaa2b4d71338620537df81b/src/ezstd.erl#L151-L169) + [C++](https://github.com/silviucpp/ezstd/blob/f3f33b2f6b917f7e8aaa2b4d71338620537df81b/c_src/ezstd_nif.cc#L520-L568) implementation for inspiration.

## 

[​

](https://docs.discord.com/developers/events/gateway#tracking-state)

Tracking State

Most of a client’s state is provided during the initial [Ready](https://docs.discord.com/developers/events/gateway#ready-event) event and in the [Guild Create](https://docs.discord.com/developers/events/gateway-events#guild-create) events that follow. As resources continue to be created, updated, and deleted, Gateway events are sent to notify the app of these changes and to provide associated data. To avoid excessive API calls, apps should cache as many relevant resource states as possible, and update them as new events are received.

For larger apps, client state can grow to be very large. Therefore, we recommend only storing data in memory that are _needed_ for the app to operate. In some cases, there isn’t a need to cache member information (like roles or permissions) since some events like [MESSAGE\_CREATE](https://docs.discord.com/developers/events/gateway-events#message-create) have the full member object included.

An example of state tracking can be considered in the case of an app that wants to track member status: when initially connecting to the Gateway, the app will receive information about the online status of guild members (whether they’re online, idle, dnd, or offline). To keep the state updated, the app will track and parse [Presence Update](https://docs.discord.com/developers/events/gateway-events#presence-update) events as they’re received, then update the cached member objects accordingly.

## 

[​

](https://docs.discord.com/developers/events/gateway#guild-availability)

Guild Availability

When connecting to the gateway as a bot user, guilds that the bot is a part of will start out as unavailable. Don’t fret! The gateway will automatically attempt to reconnect on your behalf. As guilds become available to you, you will receive [Guild Create](https://docs.discord.com/developers/events/gateway-events#guild-create) events.

## 

[​

](https://docs.discord.com/developers/events/gateway#sharding)

Sharding

As apps grow and are added to an increasing number of guilds, some developers may find it necessary to divide portions of their app’s operations across multiple processes. As such, the Gateway implements a method of user-controlled guild sharding which allows apps to split events across a number of Gateway connections. Guild sharding is entirely controlled by an app, and requires no state-sharing between separate connections to operate. While all apps _can_ enable sharding, it’s not necessary for apps in a smaller number of guilds.

Each shard can only support a maximum of 2500 guilds, and apps that are in 2500+ guilds _must_ enable sharding.

To enable sharding on a connection, the app should send the `shard` array in the [Identify](https://docs.discord.com/developers/events/gateway-events#identify) payload. The first item in this array should be the zero-based integer value of the current shard, while the second represents the total number of shards.

The [Get Gateway Bot](https://docs.discord.com/developers/events/gateway#get-gateway-bot) endpoint provides a recommended number of shards for your app in the `shards` field

To calculate which events will be sent to which shard, the following formula can be used:

###### Sharding Formula

```
shard_id = (guild_id >> 22) % num_shards
```

As an example, if you wanted to split the connection between three shards, you’d use the following values for `shard` for each connection: `[0, 3]`, `[1, 3]`, and `[2, 3]`.

Gateway events that do not contain a `guild_id` will only be sent to the first shard (`shard_id: 0`). This includes Direct Message (DM), subscription and entitlement events.

Note that `num_shards` does not relate to (or limit) the total number of potential sessions. It is only used for _routing_ traffic. As such, sessions do not have to be identified in an evenly-distributed manner when sharding. You can establish multiple sessions with the same `[shard_id, num_shards]`, or sessions with different `num_shards` values. This allows you to create sessions that will handle more or less traffic for more fine-tuned load balancing, or to orchestrate “zero-downtime” scaling/updating by handing off traffic to a new deployment of sessions with a higher or lower `num_shards` count that are prepared in parallel.

###### Max Concurrency

If you have multiple shards, you may start them concurrently based on the [`max_concurrency`](https://docs.discord.com/developers/events/gateway#session-start-limit-object) value returned to you on session start. Which shards you can start concurrently are assigned based on a key for each shard. The rate limit key for a given shard can be computed with

```
rate_limit_key = shard_id % max_concurrency
```

This puts your shards into “buckets” of `max_concurrency` size. When you start your bot, you may start up to `max_concurrency` shards at a time, and you must start them by “bucket” **in order**. To explain another way, let’s say you have 16 shards, and your `max_concurrency` is 16:

```
shard_id: 0, rate limit key (0 % 16): 0
shard_id: 1, rate limit key (1 % 16): 1
shard_id: 2, rate limit key (2 % 16): 2
shard_id: 3, rate limit key (3 % 16): 3
shard_id: 4, rate limit key (4 % 16): 4
shard_id: 5, rate limit key (5 % 16): 5
shard_id: 6, rate limit key (6 % 16): 6
shard_id: 7, rate limit key (7 % 16): 7
shard_id: 8, rate limit key (8 % 16): 8
shard_id: 9, rate limit key (9 % 16): 9
shard_id: 10, rate limit key (10 % 16): 10
shard_id: 11, rate limit key (11 % 16): 11
shard_id: 12, rate limit key (12 % 16): 12
shard_id: 13, rate limit key (13 % 16): 13
shard_id: 14, rate limit key (14 % 16): 14
shard_id: 15, rate limit key (15 % 16): 15
```

You may start all 16 of your shards at once, because each has a `rate_limit_key` which fills the bucket of 16 shards. However, let’s say you had 32 shards:

```
shard_id: 0, rate limit key (0 % 16): 0
shard_id: 1, rate limit key (1 % 16): 1
shard_id: 2, rate limit key (2 % 16): 2
shard_id: 3, rate limit key (3 % 16): 3
shard_id: 4, rate limit key (4 % 16): 4
shard_id: 5, rate limit key (5 % 16): 5
shard_id: 6, rate limit key (6 % 16): 6
shard_id: 7, rate limit key (7 % 16): 7
shard_id: 8, rate limit key (8 % 16): 8
shard_id: 9, rate limit key (9 % 16): 9
shard_id: 10, rate limit key (10 % 16): 10
shard_id: 11, rate limit key (11 % 16): 11
shard_id: 12, rate limit key (12 % 16): 12
shard_id: 13, rate limit key (13 % 16): 13
shard_id: 14, rate limit key (14 % 16): 14
shard_id: 15, rate limit key (15 % 16): 15
shard_id: 16, rate limit key (16 % 16): 0
shard_id: 17, rate limit key (17 % 16): 1
shard_id: 18, rate limit key (18 % 16): 2
shard_id: 19, rate limit key (19 % 16): 3
shard_id: 20, rate limit key (20 % 16): 4
shard_id: 21, rate limit key (21 % 16): 5
shard_id: 22, rate limit key (22 % 16): 6
shard_id: 23, rate limit key (23 % 16): 7
shard_id: 24, rate limit key (24 % 16): 8
shard_id: 25, rate limit key (25 % 16): 9
shard_id: 26, rate limit key (26 % 16): 10
shard_id: 27, rate limit key (27 % 16): 11
shard_id: 28, rate limit key (28 % 16): 12
shard_id: 29, rate limit key (29 % 16): 13
shard_id: 30, rate limit key (30 % 16): 14
shard_id: 31, rate limit key (31 % 16): 15
```

In this case, you must start the shard buckets **in “order”**. That means that you can start shard 0 -> shard 15 concurrently, and then you can start shard 16 -> shard 31.

### 

[​

](https://docs.discord.com/developers/events/gateway#sharding-for-large-bots)

Sharding for Large Bots

If your bot is in more than 150,000 guilds, there are some additional considerations you must take around sharding. Discord will migrate your bot to large bot sharding when it starts to get near the large bot sharding threshold. The bot owner(s) will receive a system DM and email confirming this move has completed as well as what shard number has been assigned. The number of shards you run must be a multiple of the shard number provided when reaching out to you. If you attempt to start your bot with an invalid number of shards, your Gateway connection will close with a `4010` Invalid Shard close code. The [Get Gateway Bot endpoint](https://docs.discord.com/developers/events/gateway#get-gateway-bot) will always return the correct amount of shards, so if you’re already using this endpoint to determine your number of shards, you shouldn’t require any changes. The session start limit for these bots will also be increased from 1000 to `max(2000, (guild_count / 1000) * 5)` per day. You also receive an increased `max_concurrency`, the number of [shards you can concurrently start](https://docs.discord.com/developers/events/gateway#session-start-limit-object).

## 

[​

](https://docs.discord.com/developers/events/gateway#get-gateway)

Get Gateway

GET/gateway

This endpoint does not require authentication.

Returns an object with a valid WSS URL which the app can use when [Connecting](https://docs.discord.com/developers/events/gateway#connecting) to the Gateway. Apps should cache this value and only call this endpoint to retrieve a new URL when they are unable to properly establish a connection using the cached one.

###### Example Response

```
{
  "url": "wss://gateway.discord.gg/"
}
```

## 

[​

](https://docs.discord.com/developers/events/gateway#get-gateway-bot)

Get Gateway Bot

GET/gateway/bot

This endpoint requires authentication using a valid bot token.

Returns an object based on the information in [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway), plus additional metadata that can help during the operation of large or [sharded](https://docs.discord.com/developers/events/gateway#sharding) bots. Unlike the [Get Gateway](https://docs.discord.com/developers/events/gateway#get-gateway), this route should not be cached for extended periods of time as the value is not guaranteed to be the same per-call, and changes as the bot joins/leaves guilds.

###### JSON Response

| Field | Type | Description |
| --- | --- | --- |
| url | string | WSS URL that can be used for connecting to the Gateway |
| shards | integer | Recommended number of [shards](https://docs.discord.com/developers/events/gateway#sharding) to use when connecting |
| session\_start\_limit | [session\_start\_limit](https://docs.discord.com/developers/events/gateway#session-start-limit-object) object | Information on the current session start limit |

###### Example Response

```
{
  "url": "wss://gateway.discord.gg/",
  "shards": 9,
  "session_start_limit": {
    "total": 1000,
    "remaining": 999,
    "reset_after": 14400000,
    "max_concurrency": 1
  }
}
```

## 

[​

](https://docs.discord.com/developers/events/gateway#session-start-limit-object)

Session Start Limit Object

###### Session Start Limit Structure

| Field | Type | Description |
| --- | --- | --- |
| total | integer | Total number of session starts the current user is allowed |
| remaining | integer | Remaining number of session starts the current user is allowed |
| reset\_after | integer | Number of milliseconds after which the limit resets |
| max\_concurrency | integer | Number of identify requests allowed per 5 seconds |

In [ ]:
#| export
class GatewayClient:
    def __init__(self, intents, client, token=None):
        self.intents,self.dc = intents,client
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.ws = self.hb_int = self.session_id = self.seq = None
        self.running = False
        gw_info = httpx.get(f'{client.base_url}/gateway/bot', headers={'Authorization': f'Bot {self.token}'}).json()
        if 'url' not in gw_info: raise ConnectionError(f"Gateway auth failed: {gw_info.get('message', gw_info)}")
        self.url = f"{gw_info['url']}?v=10&encoding=json"
        self.handlers = {'READY': self.on_rdy}
    
    async def on_rdy(self, evt):
        self.session_id, self.user_id, = evt['session_id'], evt['user']['id']
        self.resume_url = evt.get('resume_gateway_url', self.url) + '?v=10&encoding=json'
        
    def __repr__(self): return f"GatewayClient({self.intents=}, {self.url=})"

In [ ]:
# For now, let's use basic intents for guilds and messages
intents = (1 << 0) | (1 << 9) | (1 << 15)  # GUILDS | GUILD_MESSAGES | MESSAGE_CONTENT

gc = GatewayClient(intents, dc); gc

GatewayClient(self.intents=33281, self.url='wss://gateway.discord.gg?v=10&encoding=json')

`Op` is a thin helper for constructing Gateway opcodes. Discord's Gateway expects JSON payloads with an `op` field (operation code) and `d` field (payload data). Rather than hand-building dicts each time, `Op.identify()` and `Op.heartbeat()` return properly structured payloads ready to send over the WebSocket.

In [ ]:
#| export
class Op(AttrDict):
    def __repr__(self): return f"Op(op={self.op}, d={self.d})"
    @classmethod
    def identify(cls, token, intents):
        d = AttrDict(token=token, intents=intents, properties=dict(os='linux',browser='discord_wrapper', device='discord_wrapper'))
        return cls(op=2, d=d)
    @classmethod
    def heartbeat(cls, seq): return cls(op=1, d=seq)
    @classmethod
    def resume(cls, token, session_id, seq): return cls(op=6, d=AttrDict(token=token, session_id=session_id, seq=seq))

The Gateway sends events as JSON with four fields:
- **`op`** — operation code: `0` = dispatch (real events), `1` = heartbeat request, `10` = hello, `11` = heartbeat ACK
- **`t`** — event type name (only for dispatches): `MESSAGE_CREATE`, `GUILD_CREATE`, etc.
- **`s`** — sequence number (only for dispatches): increments per event, needed for heartbeats and resuming
- **`d`** — the payload data

We track the latest `s` value so heartbeats can tell Discord "I've received everything up to here." The `Event` class wraps raw JSON and auto-converts known dispatch types (like `MESSAGE_CREATE`) into their corresponding wrapper classes (`Message`, `Channel`, etc.) via `evt_typs`.

In [ ]:
#| export
evt_typs = {'MESSAGE_CREATE': Message,
            'MESSAGE_UPDATE': Message,
            'MESSAGE_DELETE': Message,
            'GUILD_CREATE': Guild,
            'CHANNEL_CREATE': Channel}

class Event(DiscordObject):
    def __init__(self, data, client):
        super().__init__(data, client)
        typ = evt_typs.get(self.t)
        self.d = typ(self.d, client) if typ else self.d
    @property
    def type(self): return self.t
    @property
    def seq(self): return self.s
    def __repr__(self): return f"Event({self.op=}, {self.type=}, {self.seq=})"

In [ ]:
#| export
@patch
async def send(self:websockets.asyncio.client.ClientConnection, msg, **kw):
    if isinstance(msg, dict): msg = json.dumps(msg)
    return await self._orig_send(msg, **kw)

`recv_evt` is the low-level building block for receiving events. It reads one raw WebSocket message, wraps it as an `Event` (auto-converting known dispatch types), and updates the sequence counter. You can use it directly to pull events one at a time — useful for debugging or understanding what Discord is sending.

In [ ]:
#| export
@patch
async def recv_evt(self:GatewayClient):
    evt = Event(json.loads(await self.ws.recv()), self.dc)
    if evt.op == 0 and evt.seq: self.seq = evt.seq
    return evt

Rather than special-casing the READY event inside `_listen`, we register it as a regular dispatch handler via `gc.on()`. When Discord confirms a successful Identify, `on_rdy` caches `session_id`, `user_id`, and `resume_gateway_url` — the three values needed to Resume after a disconnect. Note the query params appended to `resume_gateway_url` — Discord returns it bare, but the docs require the same version and encoding as the initial connection.

In [ ]:
#| export
@patch
def on(self:GatewayClient, event_type, handler): self.handlers[event_type] = handler

The gateway runs on two cooperating loops. `_hb` sends heartbeats at the interval Discord specifies and watches for ACK responses — if one is missed, the connection is zombied and gets closed with code 4000 (which preserves the session for Resuming). `_listen` reads events and dispatches them by opcode. On disconnect (exception from `recv_evt`), it calls `_reconnect` to swap in a fresh WebSocket. The new connection triggers Hello (op 10), which restarts the heartbeat and sends either Identify (first connect) or Resume (reconnect) depending on whether a session already exists. Op 7 (Reconnect) follows the same path — close and reopen.

All reconnection funnels through `_reconnect`, which closes with code 4000 (keeping the session valid) and opens a new socket to `resume_gateway_url`. The listener's op 10 handler then takes over automatically.

In [ ]:
#| export
@patch
async def _hb(self:GatewayClient):
    await asyncio.sleep(self.hb_int / 1_000 * random.random()) # jitter
    while self.running:
        self._got_ack = False
        try: await self.ws.send(Op.heartbeat(self.seq))
        except Exception: return
        await asyncio.sleep(self.hb_int / 1_000)
        if not self._got_ack: return await self.ws.close(code=4000)

In [ ]:
#| export
@patch
async def _reconnect(self:GatewayClient, resume=False):
    code, url = 4000, self.resume_url
    if not resume:
        code, url = 1000, self.url
        self.session_id = None
    await self.ws.close(code=code)
    await asyncio.sleep(2)
    self.ws = await websockets.connect(url)

In [ ]:
#| export
@patch
async def _listen(self:GatewayClient):
    while self.running:
        try:
            evt = await self.recv_evt()
            if self.debug: print('DEBUG: Received Opcode:', evt.op)
            if evt.op == 10:
                self.hb_int = evt.d['heartbeat_interval']
                if hasattr(self, '_hb_task') and self._hb_task: self._hb_task.cancel()
                self._hb_task = asyncio.create_task(self._hb())
                if self.session_id: await self.ws.send(Op.resume(self.token, self.session_id, self.seq))
                else: await self.ws.send(Op.identify(self.token, self.intents))
            elif evt.op == 0 and evt.type in self.handlers: asyncio.create_task(self.handlers[evt.type](evt.d))
            elif evt.op == 11: self._got_ack = True
            elif evt.op == 1: await self.ws.send(Op.heartbeat(self.seq))
            elif evt.op == 7: await self._reconnect()
            elif evt.op == 9: await self._reconnect(resume=evt.d)
        except Exception as e:
            print(e)
            await self._reconnect()
            continue

`start` opens the initial WebSocket and spawns the listener — nothing else. The listener handles Hello, kicks off the heartbeat, and sends Identify, so the full connection lifecycle is driven by events rather than imperative setup.

In [ ]:
#| export
@patch
async def start(self:GatewayClient, debug=False):
    self.debug = debug
    self.running = True
    self.ws = await websockets.connect(self.url)
    self._listen_task = asyncio.create_task(self._listen())

In [ ]:
await gc.start(debug=True)

DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 11


sent 4000 (private use); then received 4000 (private use)


DEBUG: Received Opcode: 10


...

sent 1000 (OK); then received 1000 (OK)


DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 0
DBuddy: Did we re-identify?


DEBUG: Received Opcode: 11


Let's now test that our gateway is getting events. Here is a simple handlers for new messages.

In [ ]:
async def on_msg(msg):
    if not msg.author.get('bot'): return
    print(f"{msg.author['username']}: {msg.content}")

gc.on('MESSAGE_CREATE', on_msg)

Now, we can send a message and it should be printed out.

In [ ]:
await ch.send('Test our event listener! Otters are awesome 🦦')

```python
Message(id=1494383536193536111, author='DBuddy', content='Test our event listener! Otter…')
```

Let's trigger and reconnection that resumes and one that doesn't and see if everything works correctly and we still end up with our printed message.

In [ ]:
await gc._reconnect(resume=True)
await asyncio.sleep(2)
await ch.send('Houston, do we have a problem?')

```python
Message(id=1494383629688901683, author='DBuddy', content='Houston, do we have a problem?')
```

In [ ]:
await gc._reconnect(resume=False)
await asyncio.sleep(2)
await ch.send('Did we re-identify?')

```python
Message(id=1494383657929019646, author='DBuddy', content='Did we re-identify?')
```

`stop` is a full shutdown — cancels both background tasks and closes the WebSocket with the default code (1000), which tells Discord to invalidate the session. After this, a fresh `start()` would need to Identify from scratch.

In [ ]:
#| export
@patch
async def stop(self:GatewayClient):
    self.running = False
    for t in (self._hb_task, self._listen_task):
        if t: t.cancel()
    if self.ws: await self.ws.close()

In [ ]:
await gc.stop()

## Voice API

In [ ]:
await gc.start()
vch = chs[3]; vch

Connected! Session: 071532487214aa5877a8dcc277014457, heartbeat: 41250ms
Gateway started!


Channel(id=1327046393453613080, name='General', type=2)

Voice requires **three simultaneous connections**:

1. **Main Gateway WebSocket** (`gc`) — to request joining a voice channel and receive voice server info
2. **Voice Gateway WebSocket** — a *separate* WebSocket to a dedicated voice server for session coordination
3. **Voice UDP** — a UDP (datagram) connection for actual audio data. UDP is used over TCP because real-time audio needs low latency more than guaranteed delivery

`VoiceClient` manages the voice gateway WebSocket and UDP connections for a single voice channel, coordinated through the main `GatewayClient`.

In [ ]:
#| export
class VoiceClient:
    def __init__(self, gateway, channel):
        self.gw, self.ch = gateway, channel
        self.ws,self.udp,self.hb_int,self.ssrc,self.secret = None,None,None,None,None
        self.running,self.v_seq,self.loop = False,-1,None
    def __repr__(self): return f'VoiceClient({self.ch=}, {self.running=})'

In [ ]:
vc = VoiceClient(gc, vch); vc

VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2), self.running=False)

Voice requires additional `Op` opcodes beyond the main gateway's identify/heartbeat. These handle the voice-specific protocol: requesting to join a channel (`voice_state`), authenticating with the voice server (`voice_identify`), negotiating encryption (`select_protocol`), signaling audio intent (`speaking`), and keeping the voice connection alive (`voice_heartbeat`).

The voice heartbeat uses a different format from the main gateway — v8 requires `seq_ack` tracking the last received sequence number. It must start **immediately** after receiving Hello, before UDP setup, or the voice WebSocket will time out (close code 4006).

In [ ]:
#| export
@patch(cls_method=True)
def voice_state(cls:Op, guild_id, channel_id):
    return cls(op=4, d=AttrDict(guild_id=guild_id, channel_id=channel_id, self_mute=False, self_deaf=False))
@patch(cls_method=True)
def voice_identify(cls:Op, server_id, user_id, session_id, token):
    return cls(op=0, d=AttrDict(server_id=server_id, user_id=user_id, session_id=session_id, token=token))
@patch(cls_method=True)
def select_protocol(cls:Op, ip, port, mode='aead_xchacha20_poly1305_rtpsize'):
    return cls(op=1, d=AttrDict(protocol='udp', data=dict(address=ip, port=port, mode=mode)))
@patch(cls_method=True)
def speaking(cls:Op, ssrc, speaking=0):
    return cls(op=5, d=AttrDict(speaking=speaking, delay=0, ssrc=ssrc))
@patch(cls_method=True)
def voice_heartbeat(cls:Op, seq_ack=-1):
    return cls(op=3, d=AttrDict(t=int(time.time() * 1000), seq_ack=seq_ack))

`_connect` handles the voice WebSocket handshake: request to join via the main gateway, wait for Discord to assign a voice server, connect to it, authenticate, and start the voice heartbeat. The heartbeat must begin immediately after Hello — before UDP setup — or the voice WebSocket times out (close code 4006).

In [ ]:
#| export
@patch
async def _vhb(self:VoiceClient):
    await asyncio.sleep(self.hb_int / 1_000 * random.random())
    while self.running:
        await self.ws.send(Op.voice_heartbeat(self.v_seq))
        await asyncio.sleep(self.hb_int / 1_000)

@patch
async def _connect(self:VoiceClient):
    self.v_srv = {}
    srv_rdy = asyncio.Event()
    async def on_vs(data):
        self.v_srv.update(data)
        srv_rdy.set()
    self.gw.on('VOICE_SERVER_UPDATE', on_vs)
    await self.gw.ws.send(Op.voice_state(self.ch.guild_id, self.ch.id))
    await srv_rdy.wait()
    self.ws = await websockets.connect(f"wss://{self.v_srv['endpoint']}?v=8")
    hello = json.loads(await self.ws.recv())
    self.hb_int = hello['d']['heartbeat_interval']
    self.running = True
    asyncio.create_task(self._vhb())
    await self.ws.send(Op.voice_identify(self.ch.guild_id, self.gw.user_id, self.gw.session_id, self.v_srv['token']))
    self.v_rdy = json.loads(await self.ws.recv())
    self.ssrc = self.v_rdy['d']['ssrc']

In [ ]:
await vc._connect(); vc

**IP Discovery**: Discord needs your *external* IP/port (what the internet sees after NAT), not your local one. We send a 74-byte UDP request containing our SSRC, and Discord responds with our external address filled in:

| Field | Size | Description |
|-------|------|-------------|
| Type | 2 bytes | 1=request, 2=response |
| Length | 2 bytes | 70 (size of remaining fields) |
| SSRC | 4 bytes | Our audio stream identifier |
| Address | 64 bytes | Blank in request; our external IP in response |
| Port | 2 bytes | Blank in request; our external port in response |

Note: Discord requires a **Speaking** event (even with `speaking=0`) before it will send audio packets to you.

In [ ]:
#| export
async def get_ip(trans, proto, ssrc):
    pkt = bytearray(74)
    pkt[0:2],pkt[2:4],pkt[4:8] = (1).to_bytes(2,'big'),(70).to_bytes(2,'big'),ssrc.to_bytes(4,'big')
    trans.sendto(pkt)
    r = await proto.packets.get()
    return r[8:72].decode('utf-8').rstrip('\x00'), int.from_bytes(r[72:74], 'big')

`_udp` sets up the audio transport: open UDP socket, discover our external IP, tell Discord which encryption mode to use (`select_protocol`), receive the secret key, and signal that we're ready to speak.

In [ ]:
#| export
class VoiceUDP(asyncio.DatagramProtocol):
    def __init__(self): self.packets = asyncio.Queue()
    def datagram_received(self, data, addr): self.packets.put_nowait(data)

In [ ]:
#| export
@patch
async def _udp(self:VoiceClient):
    udp_ip, udp_port = self.v_rdy['d']['ip'], self.v_rdy['d']['port']
    self.loop = asyncio.get_running_loop()
    self.trans, self.proto = await self.loop.create_datagram_endpoint(VoiceUDP, remote_addr=(udp_ip, udp_port))
    ip, port = await get_ip(self.trans, self.proto, self.ssrc)
    await self.ws.send(Op.select_protocol(ip, port))
    desc = json.loads(await self.ws.recv())
    while desc['op'] != 4: desc = json.loads(await self.ws.recv())
    self.secret = bytes(desc['d']['secret_key'])
    await self.ws.send(Op.speaking(self.ssrc))

In [ ]:
await vc._udp()

In [ ]:
# vc.secret

`join` ties the three phases together into a single call: connect to voice server, set up UDP, ready to receive audio.

In [ ]:
#| export
@patch
async def join(self:VoiceClient):
    await self._connect()
    await self._udp()
    print(f"Voice ready!")

In [ ]:
await vc.join()

Voice ready!


Audio arrives as UDP packets using the **RTP (Real-time Transport Protocol)** format. Each packet contains:
- **RTP header** (12+ bytes) — version, sequence number, timestamp, SSRC (identifies which user is speaking)
- **Encrypted payload** — the Opus-encoded audio, encrypted with the `secret_key`
- **Nonce** (4 bytes at end) — used for decryption

Decryption with `aead_xchacha20_poly1305_rtpsize`:
- The cipher expects a 24-byte nonce, but only 4 bytes are transmitted — we pad with 20 zero bytes
- The unencrypted RTP header is used as AAD (Additional Authenticated Data) — it's verified but not encrypted
- For `rtpsize` mode, the AAD includes the 12-byte base header, any CSRC entries (4 bytes each, usually 0), and only the 4-byte extension *preamble* — the extension *data* is part of the encrypted payload
- After decryption, we skip past the extension data (its length is in bytes 14-15 of the original packet, multiplied by 4) to reach the raw Opus audio

Packets with byte[1] == `0x78` are RTP voice data. Other packets (like `0xC9`) are RTCP control packets used for connection quality reporting — we skip those.

In [ ]:
#| export
def decrypt_pkt(pkt, secret):
    csrc_cnt = pkt[0] & 0x0F
    hdr_sz = 12 + csrc_cnt * 4 + (4 if pkt[0] & 0x10 else 0)
    nonce = bytearray(24) # only use 4, but pad to 24 bytes as expected by the cipher
    nonce[:4] = pkt[-4:]
    data, hdr, nonce = bytes(pkt[hdr_sz:-4]), bytes(pkt[:hdr_sz]), bytes(nonce)
    decrypted = xchacha_decrypt(data, hdr, nonce, bytes(secret))
    ext_sz = int.from_bytes(pkt[14:16], 'big') * 4 # extension data size in bytes
    return decrypted[ext_sz:]

Recording writes each speaker to a separate file (`recording_<ssrc>.mp3`) rather than mixing in real-time. This keeps the code simple and gives us per-speaker files — ideal for transcription. Each speaker gets their own Opus decoder because Opus is *stateful* — feeding multiple speakers through one decoder corrupts its internal state and produces garbled audio.

Silence padding ensures all files stay time-aligned: at the start (padding from `_rec_start` to first packet), and during gaps (using RTP timestamps, which Discord increments continuously even during silence). Without this, files would be compressed in time and impossible to merge or correlate.

In [ ]:
#| export
spf = 960
sr = 48_000
def silence(n_smpls:int): return b'\x00' * (n_smpls * 2) # 2 bytes per sample (s16le)

@patch
def _get_proc(self:VoiceClient, ssrc):
    if ssrc not in self._procs:
        pth = str(self._rec_path.with_stem(f"{self._rec_path.stem}_{ssrc}"))
        proc = (ffmpeg.input('pipe:', f='s16le', ar=sr, ac=1)
                      .output(pth).overwrite_output()
                      .run_async(pipe_stdin=True, quiet=True))
        proc.stdin.write(silence(int((time.time() - self._rec_start) * sr)))
        self._procs[ssrc] = proc
    return self._procs[ssrc]

@patch
def _write_audio(self:VoiceClient, ssrc, ts, data):
    proc = self._get_proc(ssrc)
    # Fill silence gaps
    if ssrc in self._last_ts:
        gap = ts - self._last_ts[ssrc] - spf
        if gap > 0: proc.stdin.write(silence(gap))
    self._last_ts[ssrc] = ts
    proc.stdin.write(self._decoders[ssrc].decode(data, spf))

@patch
async def _recv_audio(self:VoiceClient):
    while self._recording:
        try:
            pkt = await asyncio.wait_for(self.proto.packets.get(), timeout=0.1)
            if pkt[1] != 0x78: continue # filter out RTP non-voice packets 
            ssrc = int.from_bytes(pkt[8:12], 'big')
            ts = int.from_bytes(pkt[4:8], 'big')
            if ssrc not in self._decoders: self._decoders[ssrc] = opuslib_next.Decoder(sr, 1)
            if data := decrypt_pkt(pkt, self.secret): self._write_audio(ssrc, ts, data)
        except asyncio.TimeoutError: continue

`start_recording` drains any previously queued packets so recordings start clean. `stop_recording` calls `communicate()` on each ffmpeg process to flush buffered audio and cleanly close the output files — without this, recordings may be truncated or corrupted.

In [ ]:
#| export
@patch
def start_recording(self:VoiceClient, path='recording.mp3'):
    while not self.proto.packets.empty(): self.proto.packets.get_nowait()
    self._rec_path,self._decoders,self._procs,self._last_ts = Path(path),{},{},{}
    self._rec_start = time.time()
    self._recording = True
    self._rec_task = asyncio.create_task(self._recv_audio())
    return path

@patch
def stop_recording(self:VoiceClient):
    self._recording = False
    self._rec_task.cancel()
    for p in self._procs.values(): p.communicate()
    return [str(self._rec_path.with_stem(f"{self._rec_path.stem}_{ssrc}")) for ssrc in self._procs]

In [ ]:
pth = '/tmp/recording.mp3'
vc.start_recording(path=pth)
await asyncio.sleep(10)
rpths = vc.stop_recording(); rpths

In [ ]:
# Audio(rpths[0])

To leave a voice channel cleanly:
1. Stop the heartbeat loop
2. Close the Voice WebSocket
3. Close the UDP transport
4. Send a Voice State Update with `channel_id=None` on the main gateway — this tells Discord to remove the bot from the voice channel. Without this, the bot appears to stay in the channel until it times out.

In [ ]:
#| export
@patch
async def leave(self:VoiceClient):
    self.running = False
    if self.ws: await self.ws.close()
    if self.trans: self.trans.close()
    await self.gw.ws.send(Op.voice_state(self.ch.guild_id, None))

In [ ]:
await vc.leave()

In [ ]:
await gc.stop()

Gateway stopped!


## Bots

`Bot` ties together `DiscordClient` (REST) and `GatewayClient` (events) into a single object with a decorator-based command router. Commands are registered with `@bot.cmd` — the function name becomes the command name, prefixed with `!` in Discord. So `def echo` responds to `!echo`. Every command handler takes two arguments: the `Message` object and a string of everything the user typed after the command name (empty string if nothing).

The `_on_msg` handler ignores messages from the bot itself to prevent infinite loops — a common gotcha with Discord bots. It splits the message into command name + args, so `!echo hello world` passes `"hello world"` as the `args` string to the handler.

In [ ]:
#| export
class Bot:
    "Discord bot with command routing"
    def __init__(self, intents, **kw):
        self.dc = DiscordClient(**kw)
        self.gc = GatewayClient(intents, self.dc)
        self.cmds = {}
        self.errors = []
        self._on_err = None

    def cmd(self, f):
        "Register a command handler — function name becomes !name"
        self.cmds[f.__name__] = f
        return f

    async def _on_msg(self, msg):
        if msg.author['id'] == self.gc.user_id: return
        parts = msg.content.split(maxsplit=1)
        if not parts or not parts[0].startswith('!'): return
        name = parts[0][1:]
        if name not in self.cmds: return
        try:
            res = self.cmds[name](msg, parts[1] if len(parts) > 1 else '')
            if asyncio.iscoroutine(res): await res
        except Exception as e:
            self.errors.append(e)
            if self._on_err: await self._on_err(msg, e)

    async def start(self):
        "Connect to gateway and start listening"
        await self.gc.start()
        self.gc.on('MESSAGE_CREATE', self._on_msg)

    async def stop(self): await self.gc.stop()
    def __repr__(self): return f"Bot(cmds={list(self.cmds.keys())})"

In [ ]:
bot = Bot(intents)
await bot.start()

Connected! Session: aebd851f6f7e226c5f982f03cf7d27ef, heartbeat: 41250ms
Gateway started!


Commands can be registered at any time — even after `bot.start()`. This works because `@bot.cmd` just adds the function to a dict; the message handler looks up commands dynamically on each message.

In [ ]:
@bot.cmd
async def echo(msg, args): await (await msg.channel()).send(f'You said: {args}')

In [ ]:
bot

Bot(cmds=['echo'])

Errors in command handlers are caught and stored in `bot.errors` — useful for debugging in dynamic environments like solveit where you can inspect the list after the fact. For real-time handling (e.g. notifying the user in Discord), register a handler with `@bot.on_error`. Both mechanisms work simultaneously.

In [ ]:
#| export
@patch
def on_error(self:Bot, f):
    self._on_err = f
    return f

In [ ]:
@bot.on_error
async def handle_err(msg, e): print('error')

In [ ]:
@bot.cmd
def err(msg): raise Exception('test')

In [ ]:
bot.errors

[]

Voice integration reuses the existing `VoiceClient` — `Bot` just provides convenience methods to manage the lifecycle. The bot can only be in one voice channel at a time.

In [ ]:
#| export
@patch
async def join_voice(self:Bot, channel):
    "Join a voice channel and return VoiceClient"
    self.vc = VoiceClient(self.gc, channel)
    await self.vc.join()
    return self.vc

@patch
async def leave_voice(self:Bot):
    "Leave the current voice channel"
    if self.vc: await self.vc.leave()
    self.vc = None

In [ ]:
vc = await bot.join_voice(vch); vc

Voice ready!


VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2), self.running=True)

In [ ]:
vc.start_recording()

'recording.mp3'

In [ ]:
pth = vc.stop_recording()
await bot.leave_voice()

In [ ]:
# Audio(pth)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()